In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # MRI体素分类训练 - 完整12折交叉验证
# 
# 使用深度神经网络对MRI数据进行52类分类，包含背景类

# %% [markdown]
# ## 1. 导入库和设置环境

# %%
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import time
from pathlib import Path
import json
from datetime import datetime
import pandas as pd
from tqdm.notebook import tqdm
from typing import List, Tuple, Dict, Optional  # 添加类型导入

# 不需要额外导入，直接定义数据加载器

# 设置随机种子
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# %% [markdown]
# ## 2. 配置参数

# %%
# 训练配置
CONFIG = {
    'processed_dir': '/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS',
    'export_path': './models/',
    'batch_size': 512,        #  增大批次，RTX A6000可以处理
    'val_batch_size': 1024,   #  验证时用更大批次（无梯度）
    'test_batch_size': 1024,  #  测试时批次
    'no_epochs': 25,
    'no_classes': 52,
    'input_dim': 42,
    'learning_rate': 0.00001,
    'weight_decay': 0.00001,
    'dropout_rate': 0.5,
    'exclude_background': False,  # 包含背景，与原始流程一致
    'val_split': 0.1,
    'hidden_dim': 4096,
    'num_hidden_layers': 4
}

# 创建输出目录
Path(CONFIG['export_path']).mkdir(parents=True, exist_ok=True)
print("配置参数：")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

# %% [markdown]
# ## 3. 数据加载器类

# %%
class PreprocessedMRIDataset:
    """
    预处理MRI数据集加载器 - 使用flattened目录下的数据
    
    支持12折交叉验证，每个受试者作为一折
    """
    
    def __init__(self, 
                 processed_dir: str, 
                 fold: int, 
                 exclude_background: bool = False,
                 val_split: float = 0.1,
                 random_state: int = 42,
                 verbose: bool = True):
        """
        Parameters:
        -----------
        processed_dir : str
            预处理数据根目录路径
        fold : int
            当前折数 (1-12)
        exclude_background : bool
            是否排除背景体素（标签0）
        val_split : float
            验证集占训练数据的比例，默认0.1
        random_state : int
            随机种子
        verbose : bool
            是否打印详细信息
        """
        self.processed_dir = Path(processed_dir)
        self.fold = fold
        self.exclude_background = exclude_background
        self.val_split = val_split
        self.random_state = random_state
        self.verbose = verbose
        
        # 验证折数
        if not 1 <= fold <= 12:
            raise ValueError(f"fold必须在1-12之间，当前值: {fold}")
        
        # 获取所有受试者
        self.subjects = self._get_all_subjects()
        if len(self.subjects) != 12:
            raise ValueError(f"期望12个受试者，但找到{len(self.subjects)}个")
        
        # 确定数据分割
        self.test_subject = self.subjects[fold - 1]
        self.train_subjects = [s for i, s in enumerate(self.subjects) if i != fold - 1]
        
        if self.verbose:
            print(f"\n{'='*80}")
            print(f"初始化数据集 - Fold {fold}/12")
            print(f"{'='*80}")
            print(f"测试集: {self.test_subject}")
            print(f"训练/验证集: {len(self.train_subjects)}个受试者")
            print(f"排除背景: {'是' if exclude_background else '否'}")
            print(f"验证集比例: {val_split:.1%}")
    
    def _get_all_subjects(self) -> List[str]:
        """获取所有受试者ID（按名称排序）"""
        subjects = []
        for subject_dir in sorted(self.processed_dir.glob("FOR_*")):
            flattened_dir = subject_dir / "flattened"
            if flattened_dir.exists() and (flattened_dir / "features.npy").exists():
                subjects.append(subject_dir.name)
        return sorted(subjects)
    
    def _load_subject_data(self, subject_id: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        加载单个受试者的数据
        
        Returns:
        --------
        features : np.ndarray
            特征数组 (n_voxels, 42)
        labels : np.ndarray
            标签数组 (n_voxels, 52) one-hot编码
        mask : np.ndarray
            有效体素掩码 (n_voxels,) bool类型
        """
        subject_path = self.processed_dir / subject_id / "flattened"
        
        # 加载数据
        features = np.load(subject_path / "features.npy")
        
        # 优先使用one-hot编码
        if (subject_path / "labels_onehot.npy").exists():
            labels = np.load(subject_path / "labels_onehot.npy")
        else:
            # 如果没有one-hot，从映射标签创建
            if (subject_path / "labels_mapped.npy").exists():
                labels_mapped = np.load(subject_path / "labels_mapped.npy")
            else:
                labels_mapped = np.load(subject_path / "labels.npy")
                print(f" 警告: {subject_id} 使用原始标签，可能需要映射")
            
            # 转换为one-hot
            labels = self._labels_to_onehot(labels_mapped)
        
        # 创建掩码
        if self.exclude_background:
            # 背景类别是第0列
            mask = ~labels[:, 0].astype(bool)
        else:
            mask = np.ones(len(features), dtype=bool)
        
        return features, labels, mask
    
    def _labels_to_onehot(self, labels: np.ndarray, num_classes: int = 52) -> np.ndarray:
        """将标签转换为one-hot编码"""
        n_samples = labels.shape[0]
        onehot = np.zeros((n_samples, num_classes), dtype=np.float32)
        onehot[np.arange(n_samples), labels.astype(int)] = 1
        return onehot
    
    def get_fold_data(self) -> Dict[str, np.ndarray]:
        """
        获取当前折的数据
        
        Returns:
        --------
        dict : 包含以下键值的字典
            - train_data: (n_train, 42)
            - train_labels: (n_train, 52) one-hot
            - val_data: (n_val, 42)
            - val_labels: (n_val, 52) one-hot
            - test_data: (n_test, 42)
            - test_labels: (n_test, 52) one-hot
        """
        if self.verbose:
            print(f"\n加载Fold {self.fold}的数据...")
        
        start_time = time.time()
        
        # 1. 加载测试数据（单个受试者）
        test_features, test_labels, test_mask = self._load_subject_data(self.test_subject)
        test_data = test_features[test_mask]
        test_labels = test_labels[test_mask]
        
        if self.verbose:
            print(f"\n 测试集 ({self.test_subject}):")
            print(f"   • 体素数: {len(test_data):,}")
            print(f"   • 形状: 特征{test_data.shape}, 标签{test_labels.shape}")
        
        # 2. 加载训练/验证数据（其他11个受试者）
        train_features_list = []
        train_labels_list = []
        
        for subject_id in self.train_subjects:
            features, labels, mask = self._load_subject_data(subject_id)
            train_features_list.append(features[mask])
            train_labels_list.append(labels[mask])
            
            if self.verbose:
                print(f"   • {subject_id}: {mask.sum():,} 体素")
        
        # 合并所有训练数据
        all_train_features = np.vstack(train_features_list)
        all_train_labels = np.vstack(train_labels_list)
        
        # 3. 分割训练集和验证集
        if self.val_split > 0:
            # 分层采样
            train_data, val_data, train_labels, val_labels = train_test_split(
                all_train_features, 
                all_train_labels,
                test_size=self.val_split,
                random_state=self.random_state + self.fold,  # 每折使用不同种子
                stratify=np.argmax(all_train_labels, axis=1)  # 按类别分层
            )
        else:
            train_data = all_train_features
            train_labels = all_train_labels
            val_data = np.array([])
            val_labels = np.array([])
        
        load_time = time.time() - start_time
        
        if self.verbose:
            print(f"\n📈 数据集统计:")
            print(f"   • 训练集: {len(train_data):,} 体素")
            print(f"   • 验证集: {len(val_data):,} 体素")
            print(f"   • 测试集: {len(test_data):,} 体素")
            print(f"   • 总计: {len(train_data) + len(val_data) + len(test_data):,} 体素")
            print(f"   • 加载时间: {load_time:.2f}秒")
        
        return {
            'train_data': train_data.astype(np.float32),
            'train_labels': train_labels.astype(np.float32),
            'val_data': val_data.astype(np.float32),
            'val_labels': val_labels.astype(np.float32),
            'test_data': test_data.astype(np.float32),
            'test_labels': test_labels.astype(np.float32)
        }

# %% [markdown]
# ## 4. 模型定义

# %%
class RegModel(nn.Module):
    """深度全连接网络，用于MRI体素分类"""
    def __init__(self, input_dim=42, num_classes=52, hidden_dim=4096, 
                 num_hidden_layers=4, dropout_rate=0.5):
        super(RegModel, self).__init__()
        
        layers = []
        # 输入层
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        
        # 隐藏层
        for _ in range(num_hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
        
        # 输出层
        layers.append(nn.Linear(hidden_dim, num_classes))
        
        self.model = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.model(x)

# L2正则化函数
def kernel_l2_regularization(model, weight_decay=0.00001):
    """只对权重矩阵应用L2正则化，不包括偏置"""
    l2_reg = 0
    for name, param in model.named_parameters():
        if 'weight' in name and param.requires_grad:
            l2_reg += torch.norm(param, p=2) ** 2
    return weight_decay * l2_reg

# 测试模型结构
test_model = RegModel(CONFIG['input_dim'], CONFIG['no_classes'])
print(f"模型参数总数: {sum(p.numel() for p in test_model.parameters()):,}")
del test_model

# %% [markdown]
# ## 5. 数据加载函数

# %%
def load_fold_data(fold, config):
    """加载指定折的数据"""
    print(f"\n 加载 Fold {fold} 数据...")
    
    dataset = PreprocessedMRIDataset(
        processed_dir=config['processed_dir'],
        fold=fold,
        exclude_background=config['exclude_background'],
        val_split=config['val_split'],
        verbose=True
    )
    
    # 获取数据
    data = dataset.get_fold_data()
    
    # 数据标准化
    print("\n标准化数据...")
    scaler = StandardScaler()
    
    # 提取数据
    X_train = data['train_data']
    y_train = data['train_labels']
    X_val = data['val_data']
    y_val = data['val_labels']
    X_test = data['test_data']
    y_test = data['test_labels']
    
    # 标准化特征
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    
    print(f"\n数据形状:")
    print(f"  训练集: X={X_train.shape}, y={y_train.shape}")
    print(f"  验证集: X={X_val.shape}, y={y_val.shape}")
    print(f"  测试集: X={X_test.shape}, y={y_test.shape}")
    
    return X_train, y_train, X_val, y_val, X_test, y_test, scaler

# %% [markdown]
# ## 6. 评估指标函数

# %%
def calculate_metrics(y_true, y_pred, loss=None):
    """
    计算评估指标
    y_true: one-hot编码的真实标签
    y_pred: 模型预测的logits
    """
    # 转换为类别
    y_true_classes = np.argmax(y_true, axis=1)
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    # 计算macro F1
    macro_f1 = f1_score(y_true_classes, y_pred_classes, average='macro')
    
    # 计算准确率
    accuracy = np.mean(y_true_classes == y_pred_classes)
    
    metrics = {
        'accuracy': accuracy,
        'macro_f1': macro_f1
    }
    
    if loss is not None:
        metrics['loss'] = loss
    
    return metrics

# %% [markdown]
# ## 7. 训练函数

# %%
def train_epoch(model, train_loader, optimizer, criterion, device, config):
    """训练一个epoch - 正确的批次处理"""
    model.train()
    
    total_loss = 0
    all_predictions = []
    all_labels = []
    
    progress_bar = tqdm(train_loader, desc='Training', leave=False)
    
    for batch_idx, (data, target) in enumerate(progress_bar):
        #  关键：每个批次才送到GPU
        data = data.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        # 前向传播
        output = model(data)
        
        # 将one-hot编码转换为类别索引
        target_indices = torch.argmax(target, dim=1)
        
        # 计算损失
        base_loss = criterion(output, target_indices)
        l2_reg = kernel_l2_regularization(model, weight_decay=config['weight_decay'])
        loss = base_loss + l2_reg
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        #  重要：立即释放GPU内存
        total_loss += loss.item()
        all_predictions.append(output.detach().cpu().numpy())
        all_labels.append(target.detach().cpu().numpy())
        
        # 清理GPU缓存（可选，但对大数据集有帮助）
        if batch_idx % 100 == 0:
            torch.cuda.empty_cache()
        
        # 更新进度条
        progress_bar.set_postfix({'loss': loss.item(), 
                                   'GPU_mem': f'{torch.cuda.memory_allocated()/1e9:.2f}GB'})
    
    # 合并所有批次的预测
    all_predictions = np.vstack(all_predictions)
    all_labels = np.vstack(all_labels)
    
    # 计算指标
    avg_loss = total_loss / len(train_loader)
    metrics = calculate_metrics(all_labels, all_predictions, avg_loss)
    
    return metrics


def validate(model, val_data, val_labels, criterion, device, config, batch_size=32768):
    """验证模型 - 大批次处理"""
    model.eval()
    
    n_samples = len(val_data)
    all_predictions = []
    total_loss = 0
    n_batches = 0
    
    #  使用更大的批次，因为验证不需要梯度
    with torch.no_grad():
        for i in tqdm(range(0, n_samples, batch_size), desc='Validating'):
            batch_end = min(i + batch_size, n_samples)
            
            # 批次数据送GPU
            batch_data = torch.FloatTensor(val_data[i:batch_end]).to(device, non_blocking=True)
            batch_labels = torch.FloatTensor(val_labels[i:batch_end]).to(device, non_blocking=True)
            
            output = model(batch_data)
            batch_target_indices = torch.argmax(batch_labels, dim=1)
            
            # 计算损失
            base_loss = criterion(output, batch_target_indices)
            l2_reg = kernel_l2_regularization(model, weight_decay=config['weight_decay'])
            loss = base_loss + l2_reg
            
            total_loss += loss.item()
            n_batches += 1
            
            #  立即返回CPU
            all_predictions.append(output.cpu().numpy())
            
            # 清理GPU
            del batch_data, batch_labels, output
    
    # 合并所有预测
    all_predictions = np.vstack(all_predictions)
    avg_loss = total_loss / n_batches
    
    # 计算指标
    metrics = calculate_metrics(val_labels, all_predictions, avg_loss)
    
    return metrics


# %% [markdown]
# ## 8. 训练单个Fold

# %%


def train_single_fold(fold, config, verbose=True):
    """训练单个fold - 优化内存版本"""
    
    print(f"\n{'='*80}")
    print(f"训练 Fold {fold}/12")
    print(f"{'='*80}")
    
    # 加载数据到CPU内存
    X_train, y_train, X_val, y_val, X_test, y_test, scaler = load_fold_data(fold, config)
    
    #  关键修改：数据保持在CPU内存
    print("\n📌 数据策略: CPU内存 → GPU批次")
    print(f"   训练数据在CPU: {X_train.nbytes / 1e9:.2f} GB")
    print(f"   GPU批次大小: {config['batch_size']} 样本")
    print(f"   每批次GPU内存: {config['batch_size'] * 42 * 4 / 1e6:.2f} MB (特征) + "
          f"{config['batch_size'] * 52 * 4 / 1e6:.2f} MB (标签)")
    
    # 创建CPU上的tensor数据集
    X_train_tensor = torch.FloatTensor(X_train)  # CPU
    y_train_tensor = torch.FloatTensor(y_train)  # CPU
    
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    
    #  优化的DataLoader设置
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config['batch_size'], 
        shuffle=True,
        num_workers=8,           # 多进程预加载
        pin_memory=True,          # 锁页内存，加速传输
        persistent_workers=True,  # 保持worker进程
        prefetch_factor=2         # 预取批次数
    )
    
    # 创建模型（模型在GPU）
    model = RegModel(
        input_dim=config['input_dim'],
        num_classes=config['no_classes'],
        hidden_dim=config['hidden_dim'],
        num_hidden_layers=config['num_hidden_layers'],
        dropout_rate=config['dropout_rate']
    ).to(device)
    
    # 打印内存状态
    if torch.cuda.is_available():
        print(f"\n GPU内存状态:")
        print(f"   模型占用: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print(f"   可用内存: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")
    
    # 优化器和损失函数
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
    criterion = nn.CrossEntropyLoss()
    
    # 训练历史
    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    # 最佳模型跟踪
    best_val_f1 = 0
    best_epoch = 0
    best_model_state = None
    
    print("\n 开始训练...")
    start_time = time.time()
    
    # 训练循环
    for epoch in range(config['no_epochs']):
        epoch_start = time.time()
        
        # 训练
        train_metrics = train_epoch(model, train_loader, optimizer, criterion, device, config)
        
        # 验证 - 使用批次处理
        val_metrics = validate(model, X_val, y_val, criterion, device, config, 
                               batch_size=config.get('val_batch_size', 32768))
        
        # 记录历史
        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['accuracy'])
        history['train_f1'].append(train_metrics['macro_f1'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_f1'].append(val_metrics['macro_f1'])
        
        # 保存最佳模型
        if val_metrics['macro_f1'] > best_val_f1:
            best_val_f1 = val_metrics['macro_f1']
            best_epoch = epoch
            best_model_state = model.state_dict().copy()
        
        # 打印进度
        if verbose:
            print(f"\nEpoch [{epoch+1}/{config['no_epochs']}] "
                  f"Time: {time.time()-epoch_start:.2f}s")
            print(f"  Train - Loss: {train_metrics['loss']:.4f}, "
                  f"Acc: {train_metrics['accuracy']:.4f}, "
                  f"Macro F1: {train_metrics['macro_f1']:.4f}")
            print(f"  Valid - Loss: {val_metrics['loss']:.4f}, "
                  f"Acc: {val_metrics['accuracy']:.4f}, "
                  f"Macro F1: {val_metrics['macro_f1']:.4f}")
            
            # 显示GPU内存使用
            if torch.cuda.is_available():
                print(f"  GPU内存: {torch.cuda.memory_allocated()/1e9:.2f}/{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
        
        # 定期清理GPU缓存
        torch.cuda.empty_cache()
    
    training_time = time.time() - start_time
    print(f"\n 训练完成！总时间: {training_time:.2f}秒")
    print(f"最佳验证Macro F1: {best_val_f1:.4f} (Epoch {best_epoch+1})")
    
    # 加载最佳模型
    model.load_state_dict(best_model_state)
    
    # 测试集评估 - 也使用批次处理
    print("\n 测试集评估...")
    test_metrics = validate(model, X_test, y_test, criterion, device, config,
                           batch_size=config.get('test_batch_size', 32768))
    
    print(f"\n测试集结果:")
    print(f"  Loss: {test_metrics['loss']:.4f}")
    print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"  Macro F1: {test_metrics['macro_f1']:.4f}")
    
    # 保存模型
    model_path = Path(config['export_path']) / f'fold{fold}_model.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'fold': fold,
        'config': config,
        'history': history,
        'test_metrics': test_metrics,
        'best_epoch': best_epoch,
        'scaler_mean': scaler.mean_,
        'scaler_scale': scaler.scale_
    }, model_path)
    
    # 清理内存
    del X_train_tensor, y_train_tensor, train_dataset, train_loader
    torch.cuda.empty_cache()
    
    return {
        'fold': fold,
        'history': history,
        'test_metrics': test_metrics,
        'best_epoch': best_epoch,
        'training_time': training_time
    }


# %% [markdown]
# ## 9. 可视化函数

# %%
def plot_fold_history(fold_result):
    """绘制单个fold的训练历史"""
    history = fold_result['history']
    fold = fold_result['fold']
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', label='Valid Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'Fold {fold} - Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy
    ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    ax2.plot(epochs, history['val_acc'], 'r-', label='Valid Acc')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'Fold {fold} - Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Macro F1
    ax3.plot(epochs, history['train_f1'], 'b-', label='Train F1')
    ax3.plot(epochs, history['val_f1'], 'r-', label='Valid F1')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Macro F1')
    ax3.set_title(f'Fold {fold} - Macro F1')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 最终结果文本
    test_metrics = fold_result['test_metrics']
    ax4.text(0.1, 0.7, f"Fold {fold} 测试集结果:", fontsize=14, weight='bold')
    ax4.text(0.1, 0.5, f"Loss: {test_metrics['loss']:.4f}", fontsize=12)
    ax4.text(0.1, 0.4, f"Accuracy: {test_metrics['accuracy']:.4f}", fontsize=12)
    ax4.text(0.1, 0.3, f"Macro F1: {test_metrics['macro_f1']:.4f}", fontsize=12)
    ax4.text(0.1, 0.1, f"最佳Epoch: {fold_result['best_epoch']+1}", fontsize=12)
    ax4.axis('off')
    
    plt.tight_layout()
    plt.savefig(Path(CONFIG['export_path']) / f'fold{fold}_history.png', dpi=300)
    plt.show()

# %% [markdown]
# ## 10. 运行单个Fold测试

# %%
# 测试运行一个fold
test_fold = 1
result = train_single_fold(test_fold, CONFIG)
plot_fold_history(result)

# %% [markdown]
# ## 11. 完整12折交叉验证

# %%
def run_cross_validation(config):
    """运行完整的12折交叉验证"""
    
    print("开始12折交叉验证...")
    print("="*80)
    
    all_results = []
    start_time = time.time()
    
    # 创建结果汇总表
    summary_df = pd.DataFrame(columns=['Fold', 'Test Loss', 'Test Acc', 'Test Macro F1', 'Time(s)'])
    
    for fold in range(1, 13):
        try:
            result = train_single_fold(fold, config, verbose=True)
            all_results.append(result)
            
            # 添加到汇总表
            summary_df.loc[fold-1] = [
                fold,
                result['test_metrics']['loss'],
                result['test_metrics']['accuracy'],
                result['test_metrics']['macro_f1'],
                result['training_time']
            ]
            
            # 绘制训练曲线
            plot_fold_history(result)
            
        except Exception as e:
            print(f"\n Fold {fold} 训练失败: {str(e)}")
            continue
    
    total_time = time.time() - start_time
    
    # 计算统计信息
    print("\n" + "="*80)
    print(" 12折交叉验证结果汇总")
    print("="*80)
    
    print("\n各Fold结果:")
    print(summary_df.to_string(index=False, float_format='%.4f'))
    
    # 计算平均值和标准差
    mean_loss = summary_df['Test Loss'].mean()
    std_loss = summary_df['Test Loss'].std()
    mean_acc = summary_df['Test Acc'].mean()
    std_acc = summary_df['Test Acc'].std()
    mean_f1 = summary_df['Test Macro F1'].mean()
    std_f1 = summary_df['Test Macro F1'].std()
    
    print("\n统计结果:")
    print(f"  测试Loss: {mean_loss:.4f} ± {std_loss:.4f}")
    print(f"  测试准确率: {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"  测试Macro F1: {mean_f1:.4f} ± {std_f1:.4f}")
    print(f"\n总训练时间: {total_time/60:.2f} 分钟")
    print(f"平均每折时间: {summary_df['Time(s)'].mean():.2f} 秒")
    
    # 保存汇总结果
    summary_path = Path(config['export_path']) / 'cv_summary.csv'
    summary_df.to_csv(summary_path, index=False)
    
    # 保存完整结果
    results_path = Path(config['export_path']) / 'cv_results.json'
    with open(results_path, 'w') as f:
        json.dump({
            'config': config,
            'summary': {
                'mean_loss': float(mean_loss),
                'std_loss': float(std_loss),
                'mean_acc': float(mean_acc),
                'std_acc': float(std_acc),
                'mean_f1': float(mean_f1),
                'std_f1': float(std_f1)
            },
            'total_time': total_time,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }, f, indent=2)
    
    # 绘制汇总图
    plot_cv_summary(summary_df)
    
    return all_results, summary_df

def plot_cv_summary(summary_df):
    """绘制交叉验证汇总图"""
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    
    # Test Loss
    ax1.bar(summary_df['Fold'], summary_df['Test Loss'], color='skyblue', edgecolor='navy')
    ax1.axhline(y=summary_df['Test Loss'].mean(), color='red', linestyle='--', 
                label=f'Mean: {summary_df["Test Loss"].mean():.4f}')
    ax1.set_xlabel('Fold')
    ax1.set_ylabel('Test Loss')
    ax1.set_title('Test Loss by Fold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Test Accuracy
    ax2.bar(summary_df['Fold'], summary_df['Test Acc'], color='lightgreen', edgecolor='darkgreen')
    ax2.axhline(y=summary_df['Test Acc'].mean(), color='red', linestyle='--',
                label=f'Mean: {summary_df["Test Acc"].mean():.4f}')
    ax2.set_xlabel('Fold')
    ax2.set_ylabel('Test Accuracy')
    ax2.set_title('Test Accuracy by Fold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Test Macro F1
    ax3.bar(summary_df['Fold'], summary_df['Test Macro F1'], color='salmon', edgecolor='darkred')
    ax3.axhline(y=summary_df['Test Macro F1'].mean(), color='red', linestyle='--',
                label=f'Mean: {summary_df["Test Macro F1"].mean():.4f}')
    ax3.set_xlabel('Fold')
    ax3.set_ylabel('Test Macro F1')
    ax3.set_title('Test Macro F1 by Fold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(Path(CONFIG['export_path']) / 'cv_summary.png', dpi=300)
    plt.show()

# %%
# 运行完整的12折交叉验证
all_results, summary_df = run_cross_validation(CONFIG)

# %% [markdown]
# ## 12. 显著性分析（可选）

# %%
def visualize_saliency(model, input_spectrum, target_class, device):
    """计算并可视化显著性图"""
    model.eval()
    
    # 准备输入
    if input_spectrum.ndim == 1:
        input_spectrum = input_spectrum.reshape(1, -1)
    
    input_tensor = torch.FloatTensor(input_spectrum).to(device)
    input_tensor.requires_grad_(True)
    
    # 前向传播
    output = model(input_tensor)
    
    # 选择目标类别的分数
    target_score = output[0, target_class]
    
    # 计算梯度
    model.zero_grad()
    target_score.backward()
    
    # 获取显著性（梯度的绝对值）
    saliency = input_tensor.grad.data.abs().cpu().numpy()
    
    return saliency[0]

# 示例：可视化一些样本的显著性
def plot_saliency_examples(fold=1, num_examples=6):
    """绘制显著性示例"""
    
    # 加载模型和数据
    model_path = Path(CONFIG['export_path']) / f'fold{fold}_model.pth'
    checkpoint = torch.load(model_path)
    
    model = RegModel(
        input_dim=CONFIG['input_dim'],
        num_classes=CONFIG['no_classes'],
        hidden_dim=CONFIG['hidden_dim'],
        num_hidden_layers=CONFIG['num_hidden_layers'],
        dropout_rate=CONFIG['dropout_rate']
    ).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # 加载数据
    _, _, _, _, X_test, y_test, scaler = load_fold_data(fold, CONFIG)
    
    # 随机选择样本
    indices = np.random.choice(len(X_test), num_examples, replace=False)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for i, idx in enumerate(indices):
        input_spectrum = X_test[idx]
        target_class = np.argmax(y_test[idx])
        
        # 计算显著性
        saliency = visualize_saliency(model, input_spectrum, target_class, device)
        
        # 绘图
        ax = axes[i]
        ax.plot(saliency, 'g', label='Saliency', linewidth=2)
        ax.plot(input_spectrum, 'b', alpha=0.3, label='Normalized Input')
        ax.set_title(f'Class {target_class}')
        ax.set_xlabel('Feature Index')
        ax.set_ylabel('Value')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(Path(CONFIG['export_path']) / 'saliency_examples.png', dpi=300)
    plt.show()

# 运行显著性分析
plot_saliency_examples(fold=1, num_examples=6)

# %% [markdown]
# ## 13. 总结

# %%
print("\n" + "="*80)
print(" 训练完成！")
print("="*80)
print(f"\n所有结果已保存到: {CONFIG['export_path']}")
print("\n文件列表:")
for file in sorted(Path(CONFIG['export_path']).glob('*')):
    print(f"  - {file.name}")

# 显示最终汇总
print("\n最终12折交叉验证结果:")
print(f"  测试Macro F1: {summary_df['Test Macro F1'].mean():.4f} ± {summary_df['Test Macro F1'].std():.4f}")
print(f"  测试准确率: {summary_df['Test Acc'].mean():.4f} ± {summary_df['Test Acc'].std():.4f}")

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # MRI体素分类训练 - 完整12折交叉验证
# 
# 使用深度神经网络对MRI数据进行52类分类，包含背景类

# %% [markdown]
# ## 1. 导入库和设置环境

# %%
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import time
from pathlib import Path
import json
from datetime import datetime
import pandas as pd
from tqdm.notebook import tqdm
from typing import List, Tuple, Dict, Optional  # 添加类型导入

# 不需要额外导入，直接定义数据加载器

# 设置随机种子
def set_seed(seed=42):
   torch.manual_seed(seed)
   torch.cuda.manual_seed(seed)
   np.random.seed(seed)
   if torch.cuda.is_available():
       torch.cuda.manual_seed_all(seed)
   torch.backends.cudnn.deterministic = True
   torch.backends.cudnn.benchmark = False

set_seed(42)

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")
if torch.cuda.is_available():
   print(f"GPU: {torch.cuda.get_device_name(0)}")
   print(f"GPU内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# %% [markdown]
# ## 2. 配置参数

# %%
# 训练配置
CONFIG = {
   'processed_dir': '/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS',
   'export_path': './models/',
   'batch_size': 512,
   'val_batch_size': 1024,
   'test_batch_size': 1024,
   'no_epochs': 25,
   'no_classes': 52,
   'input_dim': 42,  # 原始维度保持42
   'learning_rate': 0.00001,
   'weight_decay': 0.00001,
   'dropout_rate': 0.5,
   'exclude_background': False,
   'val_split': 0.1,
   'hidden_dim': 4096,
   'num_hidden_layers': 4,
   # 特征排除配置
   'exclude_features': [14],  # 排除的特征索引列表，14是B0map
   'feature_names': [  # 记录特征名称便于追踪
       't1_mp2rage_sag_0p65_UNI-DEN_13_MR',
       'c_anatomy_nii_pre_t1_mp2rage_sag_0p65_INV1_10_MR',
       'c_anatomy_nii_pre_t1_mp2rage_sag_0p65_INV2_14_MR',
       'c_anatomy_nii_pre_t1_mp2rage_sag_0p65_T1_Images_11_MR',
       'c_anatomy_nii_pre_t2_space_dark-fluid_sag_p4_pTx_fast_1.3mm_pat9_17_MR',
       'c_anatomy_nii_pre_t2_tse_tra_seperate_18_MR',
       'c_anatomy_nii_pre_t2_tse_tra_seperate_B1filter_19_MR',
       'c_CEST_nii_post_MTR_asym_amine_2p00uT_3p0ppm_self_highre',
       'c_CEST_nii_post_MTR_asym_OH_4p00uT_1p5ppm_highre',
       'c_CEST_nii_post_PredNN_P_FOR16_1_amide_b1_p_7_popt_highre',
       'c_CEST_nii_post_PredNN_P_FOR16_1_amine_b1_p_7_popt',
       'c_CEST_nii_post_PredNN_P_FOR16_1_noe_b1_p_7_popt',
       'c_CEST_nii_post_PredNN_P_FOR16_1_ssmt_b1_p_7_popt',
       'c_CEST_nii_post_PredNN_P_FOR16_1_water_b1_p_7_popt',
       'c_qsm_nii_post_vibe_1iso_CAIPI6_8TE_RR_B0map_highre',  # 索引14 - 要排除的
       'c_qsm_nii_post_vibe_1iso_CAIPI6_8TE_RR_rot_susceptibility_highre',
       'c_qsm_nii_post_vibe_1iso_CAIPI6_8TE_RR_SMWIdiamag',
       'c_qsm_nii_post_vibe_1iso_CAIPI6_8TE_RR_SMWIparamag',
       'c_qsm_nii_post_vibe_1iso_CAIPI6_8TE_RR_T2starmap',
       'c_qsm_nii_pre_vibe_1iso_CAIPI6_8TE_RR_magnitude',
       'c_qti_nii_post_NEW_AD',
       'c_qti_nii_post_NEW_C_c',
       'c_qti_nii_post_NEW_C_M',
       'c_qti_nii_post_NEW_C_MD',
       'c_qti_nii_post_NEW_C_mu',
       'c_qti_nii_post_NEW_FA',
       'c_qti_nii_post_NEW_K_bulk_outlier=2_cx2',
       'c_qti_nii_post_NEW_K_mu_outlier=2',
       'c_qti_nii_post_NEW_K_shear_outlier=2',
       'c_qti_nii_post_NEW_MD',
       'c_qti_nii_post_NEW_MK_outlier=2',
       'c_qti_nii_post_NEW_MKt_outlier=2',
       'c_qti_nii_post_NEW_OP',
       'c_qti_nii_post_NEW_OP2',
       'c_qti_nii_post_NEW_RD',
       'c_qti_nii_post_NEW_s0',
       'c_qti_nii_post_NEW_uFA',
       'c_qti_nii_post_NEW_V_iso',
       'c_qti_nii_post_NEW_V_MD',
       'c_qti_nii_post_NEW_V_shear',
       'c_sodium_nii_pre_Sodium_LP_SenseCorrected_registered',
       'c_sodium_nii_pre_Sodium_SP_SenseCorrected_registered',
   ]
}

# 动态计算实际输入维度
actual_input_dim = CONFIG['input_dim'] - len(CONFIG.get('exclude_features', []))
CONFIG['actual_input_dim'] = actual_input_dim

# 创建输出目录
Path(CONFIG['export_path']).mkdir(parents=True, exist_ok=True)
print("配置参数：")
for key, value in CONFIG.items():
   if key == 'feature_names':
       continue  # 不打印长列表
   print(f"  {key}: {value}")

# 打印特征排除信息
if CONFIG.get('exclude_features'):
   print(f"\n特征排除配置:")
   print(f"  原始维度: {CONFIG['input_dim']}")
   print(f"  排除特征索引: {CONFIG['exclude_features']}")
   if 'feature_names' in CONFIG:
       for idx in CONFIG['exclude_features']:
           if idx < len(CONFIG['feature_names']):
               print(f"    - [{idx}] {CONFIG['feature_names'][idx]}")
   print(f"  实际输入维度: {actual_input_dim}")

# %% [markdown]
# ## 3. 数据加载器类

# %%
class PreprocessedMRIDataset:
   """
   预处理MRI数据集加载器 - 使用flattened目录下的数据
   
   支持12折交叉验证，每个受试者作为一折
   """
   
   def __init__(self, 
                processed_dir: str, 
                fold: int, 
                exclude_background: bool = False,
                val_split: float = 0.1,
                random_state: int = 42,
                verbose: bool = True):
       """
       Parameters:
       -----------
       processed_dir : str
           预处理数据根目录路径
       fold : int
           当前折数 (1-12)
       exclude_background : bool
           是否排除背景体素（标签0）
       val_split : float
           验证集占训练数据的比例，默认0.1
       random_state : int
           随机种子
       verbose : bool
           是否打印详细信息
       """
       self.processed_dir = Path(processed_dir)
       self.fold = fold
       self.exclude_background = exclude_background
       self.val_split = val_split
       self.random_state = random_state
       self.verbose = verbose
       
       # 验证折数
       if not 1 <= fold <= 12:
           raise ValueError(f"fold必须在1-12之间，当前值: {fold}")
       
       # 获取所有受试者
       self.subjects = self._get_all_subjects()
       if len(self.subjects) != 12:
           raise ValueError(f"期望12个受试者，但找到{len(self.subjects)}个")
       
       # 确定数据分割
       self.test_subject = self.subjects[fold - 1]
       self.train_subjects = [s for i, s in enumerate(self.subjects) if i != fold - 1]
       
       if self.verbose:
           print(f"\n{'='*80}")
           print(f"初始化数据集 - Fold {fold}/12")
           print(f"{'='*80}")
           print(f"测试集: {self.test_subject}")
           print(f"训练/验证集: {len(self.train_subjects)}个受试者")
           print(f"排除背景: {'是' if exclude_background else '否'}")
           print(f"验证集比例: {val_split:.1%}")
   
   def _get_all_subjects(self) -> List[str]:
       """获取所有受试者ID（按名称排序）"""
       subjects = []
       for subject_dir in sorted(self.processed_dir.glob("FOR_*")):
           flattened_dir = subject_dir / "flattened"
           if flattened_dir.exists() and (flattened_dir / "features.npy").exists():
               subjects.append(subject_dir.name)
       return sorted(subjects)
   
   def _load_subject_data(self, subject_id: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
       """
       加载单个受试者的数据
       
       Returns:
       --------
       features : np.ndarray
           特征数组 (n_voxels, 42)
       labels : np.ndarray
           标签数组 (n_voxels, 52) one-hot编码
       mask : np.ndarray
           有效体素掩码 (n_voxels,) bool类型
       """
       subject_path = self.processed_dir / subject_id / "flattened"
       
       # 加载数据
       features = np.load(subject_path / "features.npy")
       
       # 优先使用one-hot编码
       if (subject_path / "labels_onehot.npy").exists():
           labels = np.load(subject_path / "labels_onehot.npy")
       else:
           # 如果没有one-hot，从映射标签创建
           if (subject_path / "labels_mapped.npy").exists():
               labels_mapped = np.load(subject_path / "labels_mapped.npy")
           else:
               labels_mapped = np.load(subject_path / "labels.npy")
               print(f"警告: {subject_id} 使用原始标签，可能需要映射")
           
           # 转换为one-hot
           labels = self._labels_to_onehot(labels_mapped)
       
       # 创建掩码
       if self.exclude_background:
           # 背景类别是第0列
           mask = ~labels[:, 0].astype(bool)
       else:
           mask = np.ones(len(features), dtype=bool)
       
       return features, labels, mask
   
   def _labels_to_onehot(self, labels: np.ndarray, num_classes: int = 52) -> np.ndarray:
       """将标签转换为one-hot编码"""
       n_samples = labels.shape[0]
       onehot = np.zeros((n_samples, num_classes), dtype=np.float32)
       onehot[np.arange(n_samples), labels.astype(int)] = 1
       return onehot
   
   def get_fold_data(self) -> Dict[str, np.ndarray]:
       """
       获取当前折的数据
       
       Returns:
       --------
       dict : 包含以下键值的字典
           - train_data: (n_train, 42)
           - train_labels: (n_train, 52) one-hot
           - val_data: (n_val, 42)
           - val_labels: (n_val, 52) one-hot
           - test_data: (n_test, 42)
           - test_labels: (n_test, 52) one-hot
       """
       if self.verbose:
           print(f"\n加载Fold {self.fold}的数据...")
       
       start_time = time.time()
       
       # 1. 加载测试数据（单个受试者）
       test_features, test_labels, test_mask = self._load_subject_data(self.test_subject)
       test_data = test_features[test_mask]
       test_labels = test_labels[test_mask]
       
       if self.verbose:
           print(f"\n测试集 ({self.test_subject}):")
           print(f"   体素数: {len(test_data):,}")
           print(f"   形状: 特征{test_data.shape}, 标签{test_labels.shape}")
       
       # 2. 加载训练/验证数据（其他11个受试者）
       train_features_list = []
       train_labels_list = []
       
       for subject_id in self.train_subjects:
           features, labels, mask = self._load_subject_data(subject_id)
           train_features_list.append(features[mask])
           train_labels_list.append(labels[mask])
           
           if self.verbose:
               print(f"   {subject_id}: {mask.sum():,} 体素")
       
       # 合并所有训练数据
       all_train_features = np.vstack(train_features_list)
       all_train_labels = np.vstack(train_labels_list)
       
       # 3. 分割训练集和验证集
       if self.val_split > 0:
           # 分层采样
           train_data, val_data, train_labels, val_labels = train_test_split(
               all_train_features, 
               all_train_labels,
               test_size=self.val_split,
               random_state=self.random_state + self.fold,  # 每折使用不同种子
               stratify=np.argmax(all_train_labels, axis=1)  # 按类别分层
           )
       else:
           train_data = all_train_features
           train_labels = all_train_labels
           val_data = np.array([])
           val_labels = np.array([])
       
       load_time = time.time() - start_time
       
       if self.verbose:
           print(f"\n数据集统计:")
           print(f"   训练集: {len(train_data):,} 体素")
           print(f"   验证集: {len(val_data):,} 体素")
           print(f"   测试集: {len(test_data):,} 体素")
           print(f"   总计: {len(train_data) + len(val_data) + len(test_data):,} 体素")
           print(f"   加载时间: {load_time:.2f}秒")
       
       return {
           'train_data': train_data.astype(np.float32),
           'train_labels': train_labels.astype(np.float32),
           'val_data': val_data.astype(np.float32),
           'val_labels': val_labels.astype(np.float32),
           'test_data': test_data.astype(np.float32),
           'test_labels': test_labels.astype(np.float32)
       }

# %% [markdown]
# ## 4. 模型定义

# %%
class RegModel(nn.Module):
   """深度全连接网络，用于MRI体素分类"""
   def __init__(self, input_dim=42, num_classes=52, hidden_dim=4096, 
                num_hidden_layers=4, dropout_rate=0.5):
       super(RegModel, self).__init__()
       
       layers = []
       # 输入层
       layers.append(nn.Linear(input_dim, hidden_dim))
       layers.append(nn.ReLU())
       layers.append(nn.Dropout(dropout_rate))
       
       # 隐藏层
       for _ in range(num_hidden_layers - 1):
           layers.append(nn.Linear(hidden_dim, hidden_dim))
           layers.append(nn.ReLU())
           layers.append(nn.Dropout(dropout_rate))
       
       # 输出层
       layers.append(nn.Linear(hidden_dim, num_classes))
       
       self.model = nn.Sequential(*layers)
       
   def forward(self, x):
       return self.model(x)

# L2正则化函数
def kernel_l2_regularization(model, weight_decay=0.00001):
   """只对权重矩阵应用L2正则化，不包括偏置"""
   l2_reg = 0
   for name, param in model.named_parameters():
       if 'weight' in name and param.requires_grad:
           l2_reg += torch.norm(param, p=2) ** 2
   return weight_decay * l2_reg

# 测试模型结构
actual_input_dim = CONFIG.get('actual_input_dim', CONFIG['input_dim'])
test_model = RegModel(actual_input_dim, CONFIG['no_classes'])
print(f"模型参数总数: {sum(p.numel() for p in test_model.parameters()):,}")
print(f"模型输入维度: {actual_input_dim}")
print(f"模型输出维度: {CONFIG['no_classes']}")
del test_model

# %% [markdown]
# ## 5. 数据加载函数

# %%
def exclude_features(data, exclude_indices):
   """
   从数据中排除指定的特征
   
   Parameters:
   -----------
   data : np.ndarray
       形状为 (n_samples, n_features) 的数据
   exclude_indices : list
       要排除的特征索引列表
   
   Returns:
   --------
   np.ndarray : 排除指定特征后的数据
   """
   if not exclude_indices:
       return data
   
   # 创建保留特征的掩码
   all_indices = np.arange(data.shape[1])
   keep_mask = np.ones(data.shape[1], dtype=bool)
   keep_mask[exclude_indices] = False
   
   return data[:, keep_mask]

def load_fold_data(fold, config):
   """加载指定折的数据"""
   print(f"\n加载 Fold {fold} 数据...")
   
   dataset = PreprocessedMRIDataset(
       processed_dir=config['processed_dir'],
       fold=fold,
       exclude_background=config['exclude_background'],
       val_split=config['val_split'],
       verbose=True
   )
   
   # 获取数据
   data = dataset.get_fold_data()
   
   # 提取数据
   X_train = data['train_data']
   y_train = data['train_labels']
   X_val = data['val_data']
   y_val = data['val_labels']
   X_test = data['test_data']
   y_test = data['test_labels']
   
   # 排除指定特征
   if config.get('exclude_features'):
       print(f"\n排除特征索引: {config['exclude_features']}")
       X_train = exclude_features(X_train, config['exclude_features'])
       X_val = exclude_features(X_val, config['exclude_features'])
       X_test = exclude_features(X_test, config['exclude_features'])
       print(f"特征维度: {config['input_dim']} → {X_train.shape[1]}")
   
   # 数据标准化
   print("\n标准化数据...")
   scaler = StandardScaler()
   
   # 标准化特征
   X_train = scaler.fit_transform(X_train)
   X_val = scaler.transform(X_val)
   X_test = scaler.transform(X_test)
   
   print(f"\n数据形状:")
   print(f"  训练集: X={X_train.shape}, y={y_train.shape}")
   print(f"  验证集: X={X_val.shape}, y={y_val.shape}")
   print(f"  测试集: X={X_test.shape}, y={y_test.shape}")
   
   return X_train, y_train, X_val, y_val, X_test, y_test, scaler

# %% [markdown]
# ## 6. 评估指标函数

# %%
def calculate_metrics(y_true, y_pred, loss=None):
   """
   计算评估指标
   y_true: one-hot编码的真实标签
   y_pred: 模型预测的logits
   """
   # 转换为类别
   y_true_classes = np.argmax(y_true, axis=1)
   y_pred_classes = np.argmax(y_pred, axis=1)
   
   # 计算macro F1
   macro_f1 = f1_score(y_true_classes, y_pred_classes, average='macro')
   
   # 计算准确率
   accuracy = np.mean(y_true_classes == y_pred_classes)
   
   metrics = {
       'accuracy': accuracy,
       'macro_f1': macro_f1
   }
   
   if loss is not None:
       metrics['loss'] = loss
   
   return metrics

# %% [markdown]
# ## 7. 训练函数

# %%
def train_epoch(model, train_loader, optimizer, criterion, device, config):
   """训练一个epoch - 正确的批次处理"""
   model.train()
   
   total_loss = 0
   all_predictions = []
   all_labels = []
   
   progress_bar = tqdm(train_loader, desc='Training', leave=False)
   
   for batch_idx, (data, target) in enumerate(progress_bar):
       # 每个批次才送到GPU
       data = data.to(device, non_blocking=True)
       target = target.to(device, non_blocking=True)
       
       optimizer.zero_grad()
       
       # 前向传播
       output = model(data)
       
       # 将one-hot编码转换为类别索引
       target_indices = torch.argmax(target, dim=1)
       
       # 计算损失
       base_loss = criterion(output, target_indices)
       l2_reg = kernel_l2_regularization(model, weight_decay=config['weight_decay'])
       loss = base_loss + l2_reg
       
       # 反向传播
       loss.backward()
       optimizer.step()
       
       # 立即释放GPU内存
       total_loss += loss.item()
       all_predictions.append(output.detach().cpu().numpy())
       all_labels.append(target.detach().cpu().numpy())
       
       # 清理GPU缓存
       if batch_idx % 100 == 0:
           torch.cuda.empty_cache()
       
       # 更新进度条
       progress_bar.set_postfix({'loss': loss.item(), 
                                  'GPU_mem': f'{torch.cuda.memory_allocated()/1e9:.2f}GB'})
   
   # 合并所有批次的预测
   all_predictions = np.vstack(all_predictions)
   all_labels = np.vstack(all_labels)
   
   # 计算指标
   avg_loss = total_loss / len(train_loader)
   metrics = calculate_metrics(all_labels, all_predictions, avg_loss)
   
   return metrics


def validate(model, val_data, val_labels, criterion, device, config, batch_size=32768):
   """验证模型 - 大批次处理"""
   model.eval()
   
   n_samples = len(val_data)
   all_predictions = []
   total_loss = 0
   n_batches = 0
   
   # 使用更大的批次，因为验证不需要梯度
   with torch.no_grad():
       for i in tqdm(range(0, n_samples, batch_size), desc='Validating', leave=False):
           batch_end = min(i + batch_size, n_samples)
           
           # 批次数据送GPU
           batch_data = torch.FloatTensor(val_data[i:batch_end]).to(device, non_blocking=True)
           batch_labels = torch.FloatTensor(val_labels[i:batch_end]).to(device, non_blocking=True)
           
           output = model(batch_data)
           batch_target_indices = torch.argmax(batch_labels, dim=1)
           
           # 计算损失
           base_loss = criterion(output, batch_target_indices)
           l2_reg = kernel_l2_regularization(model, weight_decay=config['weight_decay'])
           loss = base_loss + l2_reg
           
           total_loss += loss.item()
           n_batches += 1
           
           # 立即返回CPU
           all_predictions.append(output.cpu().numpy())
           
           # 清理GPU
           del batch_data, batch_labels, output
   
   # 合并所有预测
   all_predictions = np.vstack(all_predictions)
   avg_loss = total_loss / n_batches
   
   # 计算指标
   metrics = calculate_metrics(val_labels, all_predictions, avg_loss)
   
   return metrics

# %% [markdown]
# ## 8. 训练单个Fold

# %%
def train_single_fold(fold, config, verbose=True):
   """训练单个fold - 优化内存版本"""
   
   print(f"\n{'='*80}")
   print(f"训练 Fold {fold}/12")
   print(f"{'='*80}")
   
   # 加载数据到CPU内存
   X_train, y_train, X_val, y_val, X_test, y_test, scaler = load_fold_data(fold, config)
   
   # 数据策略: CPU内存 → GPU批次
   print("\n数据策略: CPU内存 → GPU批次")
   print(f"  训练数据在CPU: {X_train.nbytes / 1e9:.2f} GB")
   print(f"  GPU批次大小: {config['batch_size']} 样本")
   print(f"  每批次GPU内存: {config['batch_size'] * X_train.shape[1] * 4 / 1e6:.2f} MB (特征) + "
         f"{config['batch_size'] * 52 * 4 / 1e6:.2f} MB (标签)")
   
   # 创建CPU上的tensor数据集
   X_train_tensor = torch.FloatTensor(X_train)  # CPU
   y_train_tensor = torch.FloatTensor(y_train)  # CPU
   
   train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
   
   # 优化的DataLoader设置
   train_loader = DataLoader(
       train_dataset, 
       batch_size=config['batch_size'], 
       shuffle=True,
       num_workers=8,           # 多进程预加载
       pin_memory=True,          # 锁页内存，加速传输
       persistent_workers=True,  # 保持worker进程
       prefetch_factor=2         # 预取批次数
   )
   
   # 创建模型（模型在GPU）
   # 使用实际的输入维度
   actual_input_dim = config.get('actual_input_dim', config['input_dim'])
   
   model = RegModel(
       input_dim=actual_input_dim,  # 使用实际维度
       num_classes=config['no_classes'],
       hidden_dim=config['hidden_dim'],
       num_hidden_layers=config['num_hidden_layers'],
       dropout_rate=config['dropout_rate']
   ).to(device)
   
   # 打印内存状态
   if torch.cuda.is_available():
       print(f"\nGPU内存状态:")
       print(f"  模型占用: {torch.cuda.memory_allocated()/1e9:.2f} GB")
       print(f"  可用内存: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")
   
   # 优化器和损失函数
   optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
   criterion = nn.CrossEntropyLoss()
   
   # 训练历史
   history = {
       'train_loss': [], 'train_acc': [], 'train_f1': [],
       'val_loss': [], 'val_acc': [], 'val_f1': []
   }
   
   # 最佳模型跟踪
   best_val_f1 = 0
   best_epoch = 0
   best_model_state = None
   
   print("\n开始训练...")
   start_time = time.time()
   
   # 训练循环
   for epoch in range(config['no_epochs']):
       epoch_start = time.time()
       
       # 训练
       train_metrics = train_epoch(model, train_loader, optimizer, criterion, device, config)
       
       # 验证 - 使用批次处理
       val_metrics = validate(model, X_val, y_val, criterion, device, config, 
                              batch_size=config.get('val_batch_size', 32768))
       
       # 记录历史
       history['train_loss'].append(train_metrics['loss'])
       history['train_acc'].append(train_metrics['accuracy'])
       history['train_f1'].append(train_metrics['macro_f1'])
       history['val_loss'].append(val_metrics['loss'])
       history['val_acc'].append(val_metrics['accuracy'])
       history['val_f1'].append(val_metrics['macro_f1'])
       
       # 保存最佳模型
       if val_metrics['macro_f1'] > best_val_f1:
           best_val_f1 = val_metrics['macro_f1']
           best_epoch = epoch
           best_model_state = model.state_dict().copy()
       
       # 打印进度
       if verbose:
           print(f"\nEpoch [{epoch+1}/{config['no_epochs']}] "
                 f"Time: {time.time()-epoch_start:.2f}s")
           print(f"  Train - Loss: {train_metrics['loss']:.4f}, "
                 f"Acc: {train_metrics['accuracy']:.4f}, "
                 f"Macro F1: {train_metrics['macro_f1']:.4f}")
           print(f"  Valid - Loss: {val_metrics['loss']:.4f}, "
                 f"Acc: {val_metrics['accuracy']:.4f}, "
                 f"Macro F1: {val_metrics['macro_f1']:.4f}")
           
           # 显示GPU内存使用
           if torch.cuda.is_available():
               print(f"  GPU内存: {torch.cuda.memory_allocated()/1e9:.2f}/{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
       
       # 定期清理GPU缓存
       torch.cuda.empty_cache()
   
   training_time = time.time() - start_time
   print(f"\n训练完成！总时间: {training_time:.2f}秒")
   print(f"最佳验证Macro F1: {best_val_f1:.4f} (Epoch {best_epoch+1})")
   
   # 加载最佳模型
   model.load_state_dict(best_model_state)
   
   # 测试集评估 - 也使用批次处理
   print("\n测试集评估...")
   test_metrics = validate(model, X_test, y_test, criterion, device, config,
                          batch_size=config.get('test_batch_size', 32768))
   
   print(f"\n测试集结果:")
   print(f"  Loss: {test_metrics['loss']:.4f}")
   print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
   print(f"  Macro F1: {test_metrics['macro_f1']:.4f}")
   
   # 保存模型
   model_path = Path(config['export_path']) / f'fold{fold}_model.pth'
   torch.save({
       'model_state_dict': model.state_dict(),
       'fold': fold,
       'config': config,
       'history': history,
       'test_metrics': test_metrics,
       'best_epoch': best_epoch,
       'scaler_mean': scaler.mean_,
       'scaler_scale': scaler.scale_,
       'actual_input_dim': actual_input_dim,  # 保存实际使用的输入维度
       'excluded_features': config.get('exclude_features', [])  # 保存排除的特征
   }, model_path)
   
   # 清理内存
   del X_train_tensor, y_train_tensor, train_dataset, train_loader
   torch.cuda.empty_cache()
   
   return {
       'fold': fold,
       'history': history,
       'test_metrics': test_metrics,
       'best_epoch': best_epoch,
       'training_time': training_time
   }

# %% [markdown]
# ## 9. 可视化函数

# %%
def plot_fold_history(fold_result):
   """绘制单个fold的训练历史"""
   history = fold_result['history']
   fold = fold_result['fold']
   
   fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
   epochs = range(1, len(history['train_loss']) + 1)
   
   # Loss
   ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
   ax1.plot(epochs, history['val_loss'], 'r-', label='Valid Loss')
   ax1.set_xlabel('Epoch')
   ax1.set_ylabel('Loss')
   ax1.set_title(f'Fold {fold} - Loss')
   ax1.legend()
   ax1.grid(True, alpha=0.3)
   
   # Accuracy
   ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
   ax2.plot(epochs, history['val_acc'], 'r-', label='Valid Acc')
   ax2.set_xlabel('Epoch')
   ax2.set_ylabel('Accuracy')
   ax2.set_title(f'Fold {fold} - Accuracy')
   ax2.legend()
   ax2.grid(True, alpha=0.3)
   
   # Macro F1
   ax3.plot(epochs, history['train_f1'], 'b-', label='Train F1')
   ax3.plot(epochs, history['val_f1'], 'r-', label='Valid F1')
   ax3.set_xlabel('Epoch')
   ax3.set_ylabel('Macro F1')
   ax3.set_title(f'Fold {fold} - Macro F1')
   ax3.legend()
   ax3.grid(True, alpha=0.3)
   
   # 最终结果文本
   test_metrics = fold_result['test_metrics']
   ax4.text(0.1, 0.7, f"Fold {fold} 测试集结果:", fontsize=14, weight='bold')
   ax4.text(0.1, 0.5, f"Loss: {test_metrics['loss']:.4f}", fontsize=12)
   ax4.text(0.1, 0.4, f"Accuracy: {test_metrics['accuracy']:.4f}", fontsize=12)
   ax4.text(0.1, 0.3, f"Macro F1: {test_metrics['macro_f1']:.4f}", fontsize=12)
   ax4.text(0.1, 0.1, f"最佳Epoch: {fold_result['best_epoch']+1}", fontsize=12)
   ax4.axis('off')
   
   plt.tight_layout()
   plt.savefig(Path(CONFIG['export_path']) / f'fold{fold}_history.png', dpi=300)
   plt.show()

# %% [markdown]
# ## 10. 运行单个Fold测试

# %%
# 测试运行一个fold
test_fold = 1
result = train_single_fold(test_fold, CONFIG)
plot_fold_history(result)

# %% [markdown]
# ## 11. 完整12折交叉验证

# %%
def run_cross_validation(config):
   """运行完整的12折交叉验证"""
   
   print("开始12折交叉验证...")
   print("="*80)
   
   all_results = []
   start_time = time.time()
   
   # 创建结果汇总表
   summary_df = pd.DataFrame(columns=['Fold', 'Test Loss', 'Test Acc', 'Test Macro F1', 'Time(s)'])
   
   for fold in range(1, 13):
       try:
           result = train_single_fold(fold, config, verbose=True)
           all_results.append(result)
           
           # 添加到汇总表
           summary_df.loc[fold-1] = [
               fold,
               result['test_metrics']['loss'],
               result['test_metrics']['accuracy'],
               result['test_metrics']['macro_f1'],
               result['training_time']
           ]
           
           # 绘制训练曲线
           plot_fold_history(result)
           
       except Exception as e:
           print(f"\nFold {fold} 训练失败: {str(e)}")
           continue
   
   total_time = time.time() - start_time
   
   # 计算统计信息
   print("\n" + "="*80)
   print("12折交叉验证结果汇总")
   print("="*80)
   
   print("\n各Fold结果:")
   print(summary_df.to_string(index=False, float_format='%.4f'))
   
   # 计算平均值和标准差
   mean_loss = summary_df['Test Loss'].mean()
   std_loss = summary_df['Test Loss'].std()
   mean_acc = summary_df['Test Acc'].mean()
   std_acc = summary_df['Test Acc'].std()
   mean_f1 = summary_df['Test Macro F1'].mean()
   std_f1 = summary_df['Test Macro F1'].std()
   
   print("\n统计结果:")
   print(f"  测试Loss: {mean_loss:.4f} ± {std_loss:.4f}")
   print(f"  测试准确率: {mean_acc:.4f} ± {std_acc:.4f}")
   print(f"  测试Macro F1: {mean_f1:.4f} ± {std_f1:.4f}")
   print(f"\n总训练时间: {total_time/60:.2f} 分钟")
   print(f"平均每折时间: {summary_df['Time(s)'].mean():.2f} 秒")
   
   # 保存汇总结果
   summary_path = Path(config['export_path']) / 'cv_summary.csv'
   summary_df.to_csv(summary_path, index=False)
   
   # 保存完整结果
   results_path = Path(config['export_path']) / 'cv_results.json'
   with open(results_path, 'w') as f:
       json.dump({
           'config': config,
           'summary': {
               'mean_loss': float(mean_loss),
               'std_loss': float(std_loss),
               'mean_acc': float(mean_acc),
               'std_acc': float(std_acc),
               'mean_f1': float(mean_f1),
               'std_f1': float(std_f1)
           },
           'total_time': total_time,
           'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
       }, f, indent=2)
   
   # 绘制汇总图
   plot_cv_summary(summary_df)
   
   return all_results, summary_df

def plot_cv_summary(summary_df):
   """绘制交叉验证汇总图"""
   fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
   
   # Test Loss
   ax1.bar(summary_df['Fold'], summary_df['Test Loss'], color='skyblue', edgecolor='navy')
   ax1.axhline(y=summary_df['Test Loss'].mean(), color='red', linestyle='--', 
               label=f'Mean: {summary_df["Test Loss"].mean():.4f}')
   ax1.set_xlabel('Fold')
   ax1.set_ylabel('Test Loss')
   ax1.set_title('Test Loss by Fold')
   ax1.legend()
   ax1.grid(True, alpha=0.3)
   
   # Test Accuracy
   ax2.bar(summary_df['Fold'], summary_df['Test Acc'], color='lightgreen', edgecolor='darkgreen')
   ax2.axhline(y=summary_df['Test Acc'].mean(), color='red', linestyle='--',
               label=f'Mean: {summary_df["Test Acc"].mean():.4f}')
   ax2.set_xlabel('Fold')
   ax2.set_ylabel('Test Accuracy')
   ax2.set_title('Test Accuracy by Fold')
   ax2.legend()
   ax2.grid(True, alpha=0.3)
   
   # Test Macro F1
   ax3.bar(summary_df['Fold'], summary_df['Test Macro F1'], color='salmon', edgecolor='darkred')
   ax3.axhline(y=summary_df['Test Macro F1'].mean(), color='red', linestyle='--',
               label=f'Mean: {summary_df["Test Macro F1"].mean():.4f}')
   ax3.set_xlabel('Fold')
   ax3.set_ylabel('Test Macro F1')
   ax3.set_title('Test Macro F1 by Fold')
   ax3.legend()
   ax3.grid(True, alpha=0.3)
   
   plt.tight_layout()
   plt.savefig(Path(CONFIG['export_path']) / 'cv_summary.png', dpi=300)
   plt.show()

# %%
# 运行完整的12折交叉验证
all_results, summary_df = run_cross_validation(CONFIG)

# %% [markdown]
# ## 12. 显著性分析（可选）

# %%
def visualize_saliency(model, input_spectrum, target_class, device):
   """计算并可视化显著性图"""
   model.eval()
   
   # 准备输入
   if input_spectrum.ndim == 1:
       input_spectrum = input_spectrum.reshape(1, -1)
   
   input_tensor = torch.FloatTensor(input_spectrum).to(device)
   input_tensor.requires_grad_(True)
   
   # 前向传播
   output = model(input_tensor)
   
   # 选择目标类别的分数
   target_score = output[0, target_class]
   
   # 计算梯度
   model.zero_grad()
   target_score.backward()
   
   # 获取显著性（梯度的绝对值）
   saliency = input_tensor.grad.data.abs().cpu().numpy()
   
   return saliency[0]

# 示例：可视化一些样本的显著性
def plot_saliency_examples(fold=1, num_examples=6):
   """绘制显著性示例"""
   
   # 加载模型和数据
   model_path = Path(CONFIG['export_path']) / f'fold{fold}_model.pth'
   checkpoint = torch.load(model_path)
   
   # 使用实际的输入维度
   actual_input_dim = checkpoint.get('actual_input_dim', CONFIG.get('actual_input_dim', CONFIG['input_dim']))
   
   model = RegModel(
       input_dim=actual_input_dim,  # 使用实际维度
       num_classes=CONFIG['no_classes'],
       hidden_dim=CONFIG['hidden_dim'],
       num_hidden_layers=CONFIG['num_hidden_layers'],
       dropout_rate=CONFIG['dropout_rate']
   ).to(device)
   model.load_state_dict(checkpoint['model_state_dict'])
   
   # 加载数据
   _, _, _, _, X_test, y_test, scaler = load_fold_data(fold, CONFIG)
   
   # 随机选择样本
   indices = np.random.choice(len(X_test), num_examples, replace=False)
   
   fig, axes = plt.subplots(2, 3, figsize=(18, 10))
   axes = axes.flatten()
   
   for i, idx in enumerate(indices):
       input_spectrum = X_test[idx]
       target_class = np.argmax(y_test[idx])
       
       # 计算显著性
       saliency = visualize_saliency(model, input_spectrum, target_class, device)
       
       # 绘图
       ax = axes[i]
       ax.plot(saliency, 'g', label='Saliency', linewidth=2)
       ax.plot(input_spectrum, 'b', alpha=0.3, label='Normalized Input')
       ax.set_title(f'Class {target_class}')
       ax.set_xlabel('Feature Index')
       ax.set_ylabel('Value')
       ax.legend()
       ax.grid(True, alpha=0.3)
   
   plt.tight_layout()
   plt.savefig(Path(CONFIG['export_path']) / 'saliency_examples.png', dpi=300)
   plt.show()

# 运行显著性分析
plot_saliency_examples(fold=1, num_examples=6)

# %% [markdown]
# ## 13. 总结

# %%
print("\n" + "="*80)
print("训练完成！")
print("="*80)
print(f"\n所有结果已保存到: {CONFIG['export_path']}")
print("\n文件列表:")
for file in sorted(Path(CONFIG['export_path']).glob('*')):
   print(f"  - {file.name}")

# 显示最终汇总
print("\n最终12折交叉验证结果:")
print(f"  测试Macro F1: {summary_df['Test Macro F1'].mean():.4f} ± {summary_df['Test Macro F1'].std():.4f}")
print(f"  测试准确率: {summary_df['Test Acc'].mean():.4f} ± {summary_df['Test Acc'].std():.4f}")

# 如果使用了特征排除，打印相关信息
if CONFIG.get('exclude_features'):
   print(f"\n注意: 本次训练排除了以下特征索引: {CONFIG['exclude_features']}")
   print(f"实际使用的输入维度: {CONFIG['actual_input_dim']}")